In [1]:
import pandas as pd
import json

In [2]:
import pandas as pd
import json
from pathlib import Path
import os

base_dir = Path("finetuning_eval")
json_files = list(base_dir.rglob("metrics.json"))
print(json_files)
all_dfs = []

for json_file in json_files:
    with open(json_file, "r") as f:
        data = json.load(f)

    scores_list = []
    for entry in data.get("all_primary_scores", []):
        alias, score = entry.rsplit(": ", 1)
        scores_list.append({"alias": alias, "primary_score": float(score)})
    df_primary_scores = pd.DataFrame(scores_list)

    tasks_list = []
    for task in data.get("tasks", []):
        entry = {"alias": task["alias"], "num_instances": task.get("num_instances"), "Task Suite": json_file.parts[1]}
        entry.update(task.get("metrics", {}))
        tasks_list.append(entry)
    df_tasks = pd.DataFrame(tasks_list)

    df = pd.merge(df_primary_scores, df_tasks, on="alias", how="left")
    df["model"] = os.path.basename(json_file.parent)
    all_dfs.append(df)

df = pd.concat(all_dfs, ignore_index=True)
def get_task(model_name):
    name = model_name.lower()

    if "code" in name:
        return "code"
    if "math" in name:
        return "math"
    if "sft" in name:
        return "sft"
    if "squad_v2" in name:
        return "squad_v2"
    if "ftrace" in name:
        return "ftrace"
    return "unknown"
df["model_ft_task"] = df["model"].apply(get_task)


def rename(name):
    if "arc_easy" in name:
        return "ARC\_E"
    elif "arc_challenge" in name:
        return "ARC\_C"
    elif "boolq" in name:
        return "BoolQ"
    elif "csqa" in name:
        return "CSQA"
    elif "hellaswag" in name:
        return "HSwag"
    elif "mmlu_pro" in name:
        return "MMLU-Pro"
    elif "mmlu" in name:
        return "MMLU"
    elif "openbookqa" in name:
        return "OBQA"
    elif "piqa" in name:
        return "PIQA"
    elif "socialiqa" in name:
        return "SIQA"
    elif "winogrande" in name:
        return "WinoG"
    elif "agi" in name:
        return "AGIEval"
    elif "bbh" in name:
        return "BBH"
    elif "gsm8k" in name:
        return "GSM8K"
    elif "triviaqa" in name:
        return "TriviaQA"
    elif "coqa" in name:
        return "CoQA"
    elif "drop" in name:
        return "DROP"
    elif "jeopardy" in name:
        return "JPRDY"
    elif "naturalqs" in name:
        return "NatQs"
    elif "squad_v2_conversational" in name:
        return "squad_v2_conv"
    elif "ftrace" in name:
        return "ftrace_conv"
    elif "squad" in name:
        return "SQuAD"
    elif "olmo3:base_easy:math_bpb" == name:
        return "math_bpb"
    elif "olmo3:base_easy:code_bpb" == name:
        return "code_bpb"
    elif "olmo_2_generative::olmes" == name:
        return "olmo_2_generative"
    elif "olmo_2_heldout::olmes" == name:
        return "olmo_2_heldout"
    else:
        return name+" TODO"
df

In [3]:

tasks_code= [
    "olmo3:base_easy:code_bpb"
]

tasks_math= [
    "olmo3:base_easy:math_bpb" 
]

tasks_sft= [
    "mmlu:mc::olmes",
    "olmo_2_generative::olmes",
    "olmo_2_heldout::olmes"
]
    

tasks_squad_v2 = [
    "qa:squad_v2_conversational"
]

tasks_ftrace = [
    "qa:ftrace_conversational"
]


In [ ]:
task_dfs = []
for task, benchmark_tasks in [
    ("code", tasks_code),
    ("math", tasks_math),
    ("sft", tasks_sft),
    ("squad_v2", tasks_squad_v2),
    ("ftrace", tasks_ftrace)
]:
    df_task = df[df["Task Suite"].isin(benchmark_tasks) & df["model"].str.contains(task)].copy()

    df_task["Task Suite"] = df_task["Task Suite"].apply(rename)
   
    df_task = df_task.reset_index(drop=True).groupby(["Task Suite", "model", "model_ft_task"]).mean(numeric_only=True)
    # display(df_task)
    task_dfs.append(df_task)
df_full = pd.concat(task_dfs)


In [5]:
import yaml
with open("experiments.yaml", "r") as f:
        cfg = yaml.safe_load(f)

base_models = [b.split('/')[1] for b in cfg["base_models"]]

In [6]:
base_models

In [7]:
for name, group in df_full.groupby(["model_ft_task"]):
    print(name)
    group = group["primary_score_x"].unstack("Task Suite").reset_index()
    group["Base Model"] = group["model"].apply(lambda model_name: [b for b in base_models if b in model_name ][0])
    table = group[[col for col in group.columns if col not in ["model_ft_task", "index", "model"]]].set_index("Base Model")
    styled = table.style.format_index(escape="latex", axis=1).format_index(escape="latex", axis=0).format(precision=2) 
    latex = styled.to_latex(convert_css=True, hrules=True, multicol_align="r",
                                   column_format='l' + 'l' * len(styled.data.columns))
    display(styled)
    path = f"figures/benchmark-results/{name[0]}.tex"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
                f.write(latex)
            

    # df_wide